In [3]:
%pip -q install "crewai[google-genai]==1.15.44"

from crewai import Agent, Task, Crew, Process, LLM

In [4]:
import os
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ").strip()
os.environ["GEMINI_API_KEY"] = api_key

print("Gemini API key configured successfully!")

Enter your Gemini API key: ··········
Gemini API key configured successfully!


In [5]:
llm = LLM(
    model="gemini/gemini-3.5-flash",
    temperature=0.3
)

In [6]:
coordinator = Agent(
    role="Study Coordinator",
    goal="Understand the student's learning request and organize the teaching, quiz, and evaluation process.",
    backstory=(
        "You are Leo's Study Coordinator. You identify the student's "
        "topic and learning objectives, prepare a learning plan, and "
        "coordinate the other agents. If a request is unclear, "
        "identify what needs clarification."
    ),
    llm=llm,
    verbose=True
)

In [7]:
explainer = Agent(
    role="Concept Explainer",
    goal="Teach the student's selected topic clearly using simple explanations and relevant examples.",
    backstory=(
        "You are Leo's Concept Explainer, an experienced tutor. "
        "You explain concepts step by step, provide practical examples, "
        "and adjust your explanations to the student's learning level. "
        "Follow the learning objectives provided by the Study Coordinator."
    ),
    llm=llm,
    verbose=True
)

In [8]:
quiz_master = Agent(
    role="Quiz Master",
    goal="Create structured practice questions based on the concepts taught by the Explainer.",
    backstory=(
        "You are Leo's Quiz Master, an expert in designing educational quizzes. "
        "You create three multiple-choice questions based on the Explainer's lesson. "
        "Each question must have four options, one correct answer, "
        "and a brief explanation. Keep the difficulty appropriate "
        "for the student's learning level."
    ),
    llm=llm,
    verbose=True
)

In [9]:
evaluator = Agent(
    role="Learning Evaluator",
    goal="Evaluate the student's quiz answers and provide accurate scores and constructive feedback.",
    backstory=(
        "You are Leo's Learning Evaluator, an experienced educational assessor. "
        "You compare the student's answers with the Quiz Master's correct answers, "
        "calculate the total score, and explain any mistakes. "
        "Identify concepts the student needs to review and provide "
        "encouraging, constructive feedback."
    ),
    llm=llm,
    verbose=True
)

In [10]:
coordinator_task = Task(
    description=(
        "Review the learning request from student {student_name}. "
        "The selected topic is {topic}, and the learning level is {level}. "
        "Identify three learning objectives and prepare a clear teaching plan. "
        "If the request is unclear, identify what needs clarification."
    ),
    expected_output=(
        "A concise learning plan containing the student's name, "
        "topic, learning level, three learning objectives, "
        "and the concepts that should be taught."
    ),
    agent=coordinator
)

In [11]:
explainer_task = Task(
    description=(
        "Teach the topic {topic} to student {student_name}, "
        "who has a {level} learning level. "
        "Follow the learning plan prepared by the Study Coordinator. "
        "Explain the main concepts step by step using simple language "
        "and provide at least two practical examples."
    ),
    expected_output=(
        "A structured lesson covering the Coordinator's learning objectives, "
        "clear explanations of the main concepts, "
        "and at least two practical examples."
    ),
    agent=explainer,
    context=[coordinator_task]
)

In [12]:
from pydantic import BaseModel, Field

class QuizQuestion(BaseModel):
    question: str
    options: list[str] = Field(min_length=4, max_length=4)
    correct_option: int = Field(ge=1, le=4)
    explanation: str

class QuizOutput(BaseModel):
    questions: list[QuizQuestion] = Field(min_length=3, max_length=3)

quiz_task = Task(
    description=(
        "Create exactly three multiple-choice questions about {topic} "
        "based on the lesson prepared by the Concept Explainer. "
        "Each question must have four options, one correct option "
        "numbered from 1 to 4, and a brief explanation. "
        "Use the student's {level} learning level."
    ),
    expected_output=(
        "A structured quiz containing exactly three questions. "
        "Each question must include the question, four options, "
        "the correct option number, and an explanation."
    ),
    agent=quiz_master,
    context=[explainer_task],
    output_json=QuizOutput
)

In [13]:
evaluator_task = Task(
    description=(
        "Evaluate the quiz answers submitted by student {student_name} "
        "for the topic {topic}. "
        "Quiz questions and correct answers:\n{quiz_data}\n"
        "Student's submitted answers:\n{student_answers}\n"
        "Compare each answer with the correct option. "
        "Award one point for each correct answer. "
        "Explain incorrect answers and identify concepts "
        "the student needs to review."
    ),
    expected_output=(
        "An evaluation report containing the student's score out of 3, "
        "correct and incorrect answers for each question, "
        "brief explanations of mistakes, and recommendations for improvement."
    ),
    agent=evaluator
)

In [14]:
teaching_crew = Crew(
    agents=[coordinator, explainer, quiz_master],
    tasks=[coordinator_task, explainer_task, quiz_task],
    process=Process.sequential,
    verbose=True
)

In [22]:
student_name = input("Enter your name: ").strip()

while not student_name:
    print("Coordinator: Please enter your name.")
    student_name = input("Enter your name: ").strip()

topic = input("Enter the topic you want to learn: ").strip()

while not topic:
    print("Coordinator: Please specify a topic to study.")
    topic = input("Enter the topic you want to learn: ").strip()

print(f"Coordinator: Your selected topic is '{topic}'.")

refined_topic = input(
    "Enter a more specific topic if needed (or press Enter to continue): "
).strip()

if refined_topic:
    topic = refined_topic

level = input("Learning level (beginner/intermediate/advanced): ").strip().lower()

while level not in ["beginner", "intermediate", "advanced"]:
    print("Coordinator: Please select a valid learning level.")
    level = input("Learning level (beginner/intermediate/advanced): ").strip().lower()

student_memory = {
    "student_name": student_name,
    "topic": topic,
    "level": level
}

print("Student information saved successfully!")

Enter your name: fahmid
Enter the topic you want to learn: algebra
Coordinator: Your selected topic is 'algebra'.
Enter a more specific topic if needed (or press Enter to continue): basics of algebra
Learning level (beginner/intermediate/advanced): beginner
Student information saved successfully!


In [23]:
import asyncio

teaching_result = None

try:
    teaching_result = await asyncio.wait_for(
        teaching_crew.kickoff_async(inputs=student_memory),
        timeout=300
    )
except TimeoutError:
    print("Coordinator: The teaching session took too long. Please try again.")
except Exception as error:
    print(f"Coordinator: The teaching session failed: {error}")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0e5dbc8d-d15f-4c6d-84ff-30754375562c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Review the learning request from student fahmid. The selected topic is basics of algebra, and the        │
│  learning level is beginner. Identify three learning objectives and prepare a clear teaching plan. If the       │
│  request is unclear, identify what needs clarification.                                                         │
│  ID: 294ade0c-2dd0-4f93-94ae-bc64e5cfa1eb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study Coordinator                                                                                       │
│                                                                                                                 │
│  Task: Review the learning request from student fahmid. The selected topic is basics of algebra, and the        │
│  learning level is beginner. Identify three learning objectives and prepare a clear teaching plan. If the       │
│  request is unclear, identify what needs clarification.                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study Coordinator                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Learning Plan: Basics of Algebra**                                                                           │
│                                                                                                                 │
│  **Student Name:** Fahmid                                                                                       │
│  **Topic:** Basics of Algebra                                                                                   │
│  **Learning Level:** Beginner                                                                                   │
│  **Status:** Clear (No additional clarification needed)                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### **Learning Objectives**                                                                                    │
│  1. **Understand Algebraic Fundamentals:** Identify and define variables, constants, coefficients, and          │
│  algebraic expressions.                                                                                         │
│  2. **Simplify Expressions:** Apply basic operations and the distributive property to combine like terms and    │
│  simplify algebraic expressions.                                                                                │
│  3. **Solve Simple Linear Equations:** Apply inverse operations to solve one-step and two-step single-variable  │
│  linear equations.                                                                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### **Teaching Plan & Core Concepts**                                                                          │
│                                                                                                                 │
│  #### **Module 1: Introduction to Algebraic Expressions**                                                       │
│  * **Concepts:**                                                                                                │
│    * Difference between arithmetic and algebra.                                                                 │
│    * Understanding variables (e.g., $x, y$), constants, and coefficients.                                       │
│    * Translating written words into algebraic expressions (e.g., "three more than a number" $\rightarrow x +    │
│  3$).                                                                                                           │
│    * Evaluating expressions given specific values for variables.                                                │
│                                                                                                                 │
│  #### **Module 2: Simplifying Algebraic Expressions**  

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the learning request from student fahmid. The selected topic is basics of algebra, and the        │
│  learning level is beginner. Identify three learning objectives and prepare a clear teaching plan. If the       │
│  request is unclear, identify what needs clarification.                                                         │
│  Agent: Study Coordinator                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (7 outputs left out)

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quiz Master                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  questions=[QuizQuestion(question='In the algebraic expression 4x + 5, what is the term for the number 4?',     │
│  options=['Variable', 'Constant', 'Coefficient', 'Equation'], correct_option=3, explanation='A coefficient is   │
│  the number placed directly in front of a variable that multiplies it. In 4x, 4 is the coefficient.'),          │
│  QuizQuestion(question='What is the result when you simplify the expression 3(x + 4) using the distributive     │
│  property?', options=['3x + 4', '3x + 12', 'x + 12', '7x'], correct_option=2, explanation='Using the            │
│  distributive property, multiply the number outside by every term inside: 3 multiplied by x is 3x, and 3        │
│  multiplied by 4 is 12, resulting in 3x + 12.'), QuizQuestion(question='What is the value of x in the equation  │
│  2x - 3 = 7?', options=['x = 2', 'x = 4', 'x = 5', 'x = 10'], correct_option=3, explanation='First, undo        │
│  subtraction by adding 3 to both sides to get 2x = 10. Next, undo multiplication by dividing both sides by 2,   │
│  which gives x = 5.')]                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Create exactly three multiple-choice questions about basics of algebra based on the lesson prepared by   │
│  the Concept Explainer. Each question must have four options, one correct option numbered from 1 to 4, and a    │
│  brief explanation. Use the student's beginner learning level.                                                  │
│  Agent: Quiz Master                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0e5dbc8d-d15f-4c6d-84ff-30754375562c                                                                       │
│  Final Output: {"questions":[{"question":"In the algebraic expression 4x + 5, what is the term for the number   │
│  4?","options":["Variable","Constant","Coefficient","Equation"],"correct_option":3,"explanation":"A             │
│  coefficient is the number placed directly in front of a variable that multiplies it. In 4x, 4 is the           │
│  coefficient."},{"question":"What is the result when you simplify the expression 3(x + 4) using the             │
│  distributive property?","options":["3x + 4","3x + 12","x + 12","7x"],"correct_option":2,"explanation":"Using   │
│  the distributive property, multiply the number outside by every term inside: 3 multiplied by x is 3x, and 3    │
│  multiplied by 4 is 12, resulting in 3x + 12."},{"question":"What is the value of x in the equation 2x - 3 =    │
│  7?","options":["x = 2","x = 4","x = 5","x = 10"],"correct_option":3,"explanation":"First, undo subtraction by  │
│  adding 3 to both sides to get 2x = 10. Next, undo multiplication by dividing both sides by 2, which gives x =  │
│  5."}]}                                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [24]:
if teaching_result is None or not teaching_result.json_dict:
    print("Coordinator: No quiz is available. Please restart the teaching session.")
else:
    quiz_data = teaching_result.json_dict
    student_answers = []

    for i, question in enumerate(quiz_data["questions"], start=1):
        print(f"\nQuestion {i}: {question['question']}")

        for j, option in enumerate(question["options"], start=1):
            print(f"{j}. {option}")

        while True:
            answer = input("Your answer (1-4): ").strip()

            if answer in ["1", "2", "3", "4"]:
                student_answers.append(int(answer))
                break

            print("Please enter a number from 1 to 4.")

    print("\nStudent answers recorded successfully!")


Question 1: In the algebraic expression 4x + 5, what is the term for the number 4?
1. Variable
2. Constant
3. Coefficient
4. Equation
Your answer (1-4): 3

Question 2: What is the result when you simplify the expression 3(x + 4) using the distributive property?
1. 3x + 4
2. 3x + 12
3. x + 12
4. 7x
Your answer (1-4): 2

Question 3: What is the value of x in the equation 2x - 3 = 7?
1. x = 2
2. x = 4
3. x = 5
4. x = 10
Your answer (1-4): 4

Student answers recorded successfully!


In [25]:
import json

evaluation_result = None

if teaching_result is None or not teaching_result.json_dict:
    print("Coordinator: Evaluation unavailable because teaching did not complete.")

elif (
    "quiz_data" not in globals()
    or "student_answers" not in globals()
    or quiz_data != teaching_result.json_dict
    or len(student_answers) != len(quiz_data["questions"])
):
    print("Coordinator: Please complete the current quiz before evaluation.")

else:
    evaluation_crew = Crew(
        agents=[evaluator],
        tasks=[evaluator_task],
        process=Process.sequential,
        verbose=True
    )

    try:
        evaluation_result = await asyncio.wait_for(
            evaluation_crew.kickoff_async(
                inputs={
                    "student_name": student_memory["student_name"],
                    "topic": student_memory["topic"],
                    "quiz_data": json.dumps(quiz_data),
                    "student_answers": json.dumps(student_answers)
                }
            ),
            timeout=300
        )

        print(evaluation_result.raw)

    except TimeoutError:
        print("Coordinator: Evaluation timed out. Please try again.")

    except Exception as error:
        print(f"Coordinator: Evaluation failed: {error}")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 59d71cab-de78-47f1-bb8d-1d26eaa91067                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Evaluate the quiz answers submitted by student fahmid for the topic basics of algebra. Quiz questions    │
│  and correct answers:                                                                                           │
│  {"questions": [{"question": "In the algebraic expression 4x + 5, what is the term for the number 4?",          │
│  "options": ["Variable", "Constant", "Coefficient", "Equation"], "correct_option": 3, "explanation": "A         │
│  coefficient is the number placed directly in front of a variable that multiplies it. In 4x, 4 is the           │
│  coefficient."}, {"question": "What is the result when you simplify the expression 3(x + 4) using the           │
│  distributive property?", "options": ["3x + 4", "3x + 12", "x + 12", "7x"], "correct_option": 2,                │
│  "explanation": "Using the distributive property, multiply the number outside by every term inside: 3           │
│  multiplied by x is 3x, and 3 multiplied by 4 is 12, resulting in 3x + 12."}, {"question": "What is the value   │
│  of x in the equation 2x - 3 = 7?", "options": ["x = 2", "x = 4", "x = 5", "x = 10"], "correct_option": 3,      │
│  "explanation": "First, undo subtraction by adding 3 to both sides to get 2x = 10. Next, undo multiplication    │
│  by dividing both sides by 2, which gives x = 5."}]}                                                            │
│  Student's submitted answers:                                                                                   │
│  [3, 2, 4]                                                                                                      │
│  Compare each answer with the correct option. Award one point for each correct answer. Explain incorrect        │
│  answers and identify concepts the student needs to review.                                                     │
│  ID: fd5d34e3-4ea6-4756-8b0a-7f426d621a0f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Learning Evaluator                                                                                      │
│                                                                                                                 │
│  Task: Evaluate the quiz answers submitted by student fahmid for the topic basics of algebra. Quiz questions    │
│  and correct answers:                                                                                           │
│  {"questions": [{"question": "In the algebraic expression 4x + 5, what is the term for the number 4?",          │
│  "options": ["Variable", "Constant", "Coefficient", "Equation"], "correct_option": 3, "explanation": "A         │
│  coefficient is the number placed directly in front of a variable that multiplies it. In 4x, 4 is the           │
│  coefficient."}, {"question": "What is the result when you simplify the expression 3(x + 4) using the           │
│  distributive property?", "options": ["3x + 4", "3x + 12", "x + 12", "7x"], "correct_option": 2,                │
│  "explanation": "Using the distributive property, multiply the number outside by every term inside: 3           │
│  multiplied by x is 3x, and 3 multiplied by 4 is 12, resulting in 3x + 12."}, {"question": "What is the value   │
│  of x in the equation 2x - 3 = 7?", "options": ["x = 2", "x = 4", "x = 5", "x = 10"], "correct_option": 3,      │
│  "explanation": "First, undo subtraction by adding 3 to both sides to get 2x = 10. Next, undo multiplication    │
│  by dividing both sides by 2, which gives x = 5."}]}                                                            │
│  Student's submitted answers:                                                                                   │
│  [3, 2, 4]                                                                                                      │
│  Compare each answer with the correct option. Award one point for each correct answer. Explain incorrect        │
│  answers and identify concepts the student needs to review.                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Learning Evaluator                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Quiz Evaluation Report                                                                                       │
│                                                                                                                 │
│  **Student Name:** Fahmid                                                                                       │
│  **Topic:** Basics of Algebra                                                                                   │
│  **Evaluator:** Leo's Learning Evaluator                                                                        │
│  **Total Score:** 2 / 3 (66.7%)                                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Detailed Question Breakdown                                                                                │
│                                                                                                                 │
│  #### **Question 1: Identification of Algebraic Terms**                                                         │
│  * **Question:** In the algebraic expression $4x + 5$, what is the term for the number $4$?                     │
│  * **Student Answer:** Option 3 ("Coefficient")                                                                 │
│  * **Correct Answer:** Option 3 ("Coefficient")                                                                 │
│  * **Status:** **Correct (+1 point)**                                                                           │
│  * **Feedback:** Excellent work! You correctly identified that a number multiplying a variable is called a      │
│  coefficient.                                                                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  #### **Question 2: Distributive Property**                                                                     │
│  * **Question:** What is the result when you simplify the expression $3(x + 4)$ using the distributive          │
│  property?                                                                                                      │
│  * **Student Answer:** Option 2 ("$3x + 12$")                                                                   │
│  * **Correct Answer:** Option 2 ("$3x + 12$")                                                                   │
│  * **Status:** **Correct (+1 point)**                                                                           │
│  * **Feedback:** Great job! You properly applied the distributive property by multiplying $3$ by both $x$ and   │
│  $4$.                                                                                                           │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Evaluate the quiz answers submitted by student fahmid for the topic basics of algebra. Quiz questions    │
│  and correct answers:                                                                                           │
│  {"questions": [{"question": "In the algebraic expression 4x + 5, what is the term for the number 4?",          │
│  "options": ["Variable", "Constant", "Coefficient", "Equation"], "correct_option": 3, "explanation": "A         │
│  coefficient is the number placed directly in front of a variable that multiplies it. In 4x, 4 is the           │
│  coefficient."}, {"question": "What is the result when you simplify the expression 3(x + 4) using the           │
│  distributive property?", "options": ["3x + 4", "3x + 12", "x + 12", "7x"], "correct_option": 2,                │
│  "explanation": "Using the distributive property, multiply the number outside by every term inside: 3           │
│  multiplied by x is 3x, and 3 multiplied by 4 is 12, resulting in 3x + 12."}, {"question": "What is the value   │
│  of x in the equation 2x - 3 = 7?", "options": ["x = 2", "x = 4", "x = 5", "x = 10"], "correct_option": 3,      │
│  "explanation": "First, undo subtraction by adding 3 to both sides to get 2x = 10. Next, undo multiplication    │
│  by dividing both sides by 2, which gives x = 5."}]}                                                            │
│  Student's submitted answers:                                                                                   │
│  [3, 2, 4]                                                                                                      │
│  Compare each answer with the correct option. Award one point for each correct answer. Explain incorrect        │
│  answers and identify concepts the student needs to review.                                                     │
│  Agent: Learning Evaluator                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 59d71cab-de78-47f1-bb8d-1d26eaa91067                                                                       │
│  Final Output: # Quiz Evaluation Report                                                                         │
│                                                                                                                 │
│  **Student Name:** Fahmid                                                                                       │
│  **Topic:** Basics of Algebra                                                                                   │
│  **Evaluator:** Leo's Learning Evaluator                                                                        │
│  **Total Score:** 2 / 3 (66.7%)                                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Detailed Question Breakdown                                                                                │
│                                                                                                                 │
│  #### **Question 1: Identification of Algebraic Terms**                                                         │
│  * **Question:** In the algebraic expression $4x + 5$, what is the term for the number $4$?                     │
│  * **Student Answer:** Option 3 ("Coefficient")                                                                 │
│  * **Correct Answer:** Option 3 ("Coefficient")                                                                 │
│  * **Status:** **Correct (+1 point)**                                                                           │
│  * **Feedback:** Excellent work! You correctly identified that a number multiplying a variable is called a      │
│  coefficient.                                                                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  #### **Question 2: Distributive Property**                                                                     │
│  * **Question:** What is the result when you simplify the expression $3(x + 4)$ using the distributive          │
│  property?                                                                                                      │
│  * **Student Answer:** Option 2 ("$3x + 12$")                                                                   │
│  * **Correct Answer:** Option 2 ("$3x + 12$")                                                                   │
│  * **Status:** **Correct (+1 point)**                                                                           │
│  * **Feedback:** Great job! You properly applied the distributive property by multiplying $3$ by both $x$ and   │
│  $4$.                                                                                                           │
│                                                       

# Quiz Evaluation Report

**Student Name:** Fahmid  
**Topic:** Basics of Algebra  
**Evaluator:** Leo's Learning Evaluator  
**Total Score:** 2 / 3 (66.7%)

---

### Detailed Question Breakdown

#### **Question 1: Identification of Algebraic Terms**
* **Question:** In the algebraic expression $4x + 5$, what is the term for the number $4$?
* **Student Answer:** Option 3 ("Coefficient")
* **Correct Answer:** Option 3 ("Coefficient")
* **Status:** **Correct (+1 point)**
* **Feedback:** Excellent work! You correctly identified that a number multiplying a variable is called a coefficient.

---

#### **Question 2: Distributive Property**
* **Question:** What is the result when you simplify the expression $3(x + 4)$ using the distributive property?
* **Student Answer:** Option 2 ("$3x + 12$")
* **Correct Answer:** Option 2 ("$3x + 12$")
* **Status:** **Correct (+1 point)**
* **Feedback:** Great job! You properly applied the distributive property by multiplying $3$ by both $x$ and $4$.

---



In [26]:
if evaluation_result is None:
    print("Coordinator: Evaluation was unsuccessful. Student progress was not updated.")

else:
    student_memory["student_answers"] = student_answers

    student_memory["quiz_score"] = sum(
        answer == question["correct_option"]
        for answer, question in zip(student_answers, quiz_data["questions"])
    )

    student_memory["feedback"] = evaluation_result.raw

    print("Student:", student_memory["student_name"])
    print("Topic:", student_memory["topic"])
    print("Score:", student_memory["quiz_score"], "/", len(quiz_data["questions"]))
    print("Student memory updated successfully!")

Student: fahmid
Topic: basics of algebra
Score: 2 / 3
Student memory updated successfully!
